# **ST 554 Final Project**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

## **Setup**

The code below imports the required modules that are needed for this project.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, OneHotEncoder, PCA, \
                               StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.ml import Pipeline

#from sklearn import linear_model
#from math import sqrt
#from sklearn.model_selection import train_test_split, cross_validate
#from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
#from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge, ElasticNet, \
                                 #LogisticRegressionCV, LassoCV, RidgeCV, ElasticNetCV
#from sklearn.preprocessing import PolynomialFeatures

#from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
#from sklearn.model_selection import GridSearchCV
#from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
#from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

The code below sets up our `spark` object and only allows errors to print out in the future (i.e., it suppresses warnings).

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 16:28:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 16:28:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 16:28:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 16:28:35 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


The code below reads in the `power_ml_data.csv` file from the provided URL using `pandas`.

In [3]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe. This new object will be called `power_ML`. For better readability, throughout this project, I will display results using `.toPandas()` and `.head()` rather than `.show()`. It is important to note that this choice does not change `power_ML` back to a `pandas` or `pandas-on-spark` dataframe; rather, it just changes the way it is *displayed*.

In [5]:
power_ML = spark.createDataFrame(power_data)
power_ML.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


We are going to treat the `Power_Zone_3` variable as our response variable. We want to imagine that we know that the `Power_Zone_3` reading is going to go offline in the future, and we need to be able to predict that value appropriately using the other variables in the dataset.

## **Fitting the Model**

We are going to fit an elastic net model using CV without a training & testing split. Before we can fit our model, we have to perform the required transformations using `MLlib` functions. All (nested) transformations will be saved as objects so 1) We can display our progress through the transformations and 2) We can check to make sure we do not miss any transformations in the final pipeline.

First, we need to check to see how the `Hour` column is stored. 

In [6]:
power_ML.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

Now that we have confirmed that it's not a `DoubleType`, we will need to use an SQL transformer to cast the `Hour` variable as a `DoubleType`.

In [7]:
cast_hour = SQLTransformer(
    statement = """
                SELECT *, CAST(Hour as DOUBLE) as Hour_Double_Type FROM __THIS__
                """
            )

tf_1 = cast_hour.transform(power_ML)
tf_1.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0


Next, we need to binarize the updated `Hour_Double_Type` column based on it being less than 6.5 or not. We are essentially creating a night vs. day variable.

In [8]:
binarize_hour = Binarizer(threshold = 6.5, inputCol = "Hour_Double_Type", outputCol = "Night_vs_Day")

tf_2 = binarize_hour.transform(tf_1)
tf_2.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0


Continuing on, we need to one-hot encode the `Month` variable. Since `Month` is already an integer, we do _**not**_ need to use `StringIndexer()`.

In [10]:
encoder_OHE = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_OHE"])
model_OHE = encoder_OHE.fit(tf_2)

tf_3 = model_OHE.transform(tf_2)
tf_3.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Our next step is to run a PCA fit on the following variables:
- `Temperature`
- `Humidity`
- `Wind_Speed`
- `General_Diffuse_Flows`
- `Diffuse_Flows`

We will also need to standarize our predictors using `StandardScaler()`.

**Note:** This will be a multi-step process, outlined with in-line comments in the code chunk below.

In [12]:
# use VectorAssembler() to place the above variables in a column together for use with the PCA() estimator
assembler_PCA = VectorAssembler(
                inputCols = ["Temperature", "Humidity", "Wind_Speed",
                             "General_Diffuse_Flows", "Diffuse_Flows"],
                outputCol = "features_for_PCA",
                handleInvalid = "keep"
             )

# update transformations
tf_4 = assembler_PCA.transform(tf_3)

# use StandardScaler() to standardize predictors
scaler_PCA = StandardScaler(inputCol = "features_for_PCA", outputCol = "features_for_PCA_scaled",
                            withStd = True, withMean = True)
# fit scaler_PCA
scaler_fit = scaler_PCA.fit(tf_4)

# update transformations
tf_5 = scaler_fit.transform(tf_4)

# run PCA, using 2 PC's in the transformation
PCA_func = PCA(k = 2, inputCol = "features_for_PCA_scaled", outputCol = "PCA_features_scaled")

# fit the PCA model
PCA_model = PCA_func.fit(tf_5)

# update transformations
tf_6 = PCA_model.transform(tf_5)

# show updated transformations
tf_6.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463723, 0.8078678829472257]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806544, 0.8152476222806385]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.1120662633791016, 0.8227151924829952]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.14350318474222, 0.8329135817940956]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.182444003662701, 0.8444681624812326]"


We now need to put our model predictors into a single column called features. We can do this with `VectorAssembler()`. We will also use `StandardScaler()` to standardize our features. Since `PCA_features_scaled` has already been standardized, we do not need to re-standardize it, so this will be a multi-step process.

In [13]:
# assemble features vector without PCA for scaling
assembler_features = VectorAssembler(
                          inputCols = ["Night_vs_Day", "Power_Zone_1",
                                       "Power_Zone_2", "Month_OHE"],
                          outputCol = "features_pre_scale",
                          handleInvalid = "keep"
                    )

# update transformations
tf_7 = assembler_features.transform(tf_6)

# use StandardScaler() to standardize features -- remember PCA_features_scaled is already standardized!
scaler_features = StandardScaler(inputCol = "features_pre_scale", outputCol = "features_scaled",
                                 withStd = True, withMean = True)

# fit scaler_features
scaler_fit_feat = scaler_features.fit(tf_7)

# update transformations
tf_8 = scaler_fit_feat.transform(tf_7)

# use VectorAssembler() again to combine PCA_features_scaled and features_scaled into one features column
assembler_features_final = VectorAssembler(
                                inputCols = ["PCA_features_scaled", "features_scaled"],
                                outputCol = "features",
                                handleInvalid = "keep"
                        )

# update transformations
tf_9 = assembler_features_final.transform(tf_8)

# show updated transformations
tf_9.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled,features_pre_scale,features_scaled,features
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463723, 0.8078678829472257]","(0.0, 34055.6962, 16128.87538, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, 0.24130775579578978, -0....","[2.0690743213463723, 0.8078678829472257, -1.55..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806544, 0.8152476222806385]","(0.0, 29814.68354, 19375.07599, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.35350356900601565, -0...","[2.1029210638806544, 0.8152476222806385, -1.55..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.1120662633791016, 0.8227151924829952]","(0.0, 29128.10127, 19006.68693, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.449798237837335, -0.3...","[2.1120662633791016, 0.8227151924829952, -1.55..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.14350318474222, 0.8329135817940956]","(0.0, 28228.86076, 18361.09422, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.5759186911227896, -0....","[2.14350318474222, 0.8329135817940956, -1.5544..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.182444003662701, 0.8444681624812326]","(0.0, 27335.6962, 17872.34043, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, -0.7011869790980542, -0....","[2.182444003662701, 0.8444681624812326, -1.554..."


Finally, we rename the response variable, `Power_Zone_3`, as `label`. We will also use the SQL transformer to select the `features` vector that was created in the previous step.

In [14]:
label_and_features = SQLTransformer(
    statement = """
                SELECT features, Power_Zone_3 as label FROM __THIS__
                """
            )

# update transformations
tf_10 = label_and_features.transform(tf_9)
tf_10.toPandas().head()

,features,label
0,"[2.0690743213463723, 0.8078678829472257, -1.55...",20240.96386
1,"[2.1029210638806544, 0.8152476222806385, -1.55...",20131.08434
2,"[2.1120662633791016, 0.8227151924829952, -1.55...",19668.43373
3,"[2.14350318474222, 0.8329135817940956, -1.5544...",18899.27711
4,"[2.182444003662701, 0.8444681624812326, -1.554...",18442.40964


We can now put all of our transformations into the pipeline. We will have 10 of them, plus our linear regression instance. They are:
- `cast_hour`
- `binarize_hour`
- `encoder_OHE`
- `assembler_PCA`
- `scaler_PCA`
- `PCA_func`
- `assembler_features`
- `scaler_features`
- `assembler_features_final`
- `label_and_features`
- and `lr`, which will be the instance of our `LinearRegression()`

**Note:** Only the transformers and estimators need to be put into the pipeline since the pipeline will automatically handle which ones need to be fit. The fit objects up to this point were only created to show the progress of the transformations and check their functionality, but they themselves do not go into the pipeline!

*Example:* `encoder_OHE` *goes into the pipeline rather than* `model_OHE`

In [15]:
lr = LinearRegression()
pipeline = Pipeline(stages = [cast_hour, binarize_hour, encoder_OHE, assembler_PCA,
                              scaler_PCA, PCA_func, assembler_features, scaler_features,
                              assembler_features_final, label_and_features, lr])

We are now ready to fit our elastic net model. We are provided with the following grid for `regParam` and `elasticNetParam`:
- `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

Since our `pipeline` object is already set up, we have the following remaining steps to complete in creating our model:
- build the parameter grid
- use cross-validation (5 folds) to choose the best combination of parameters using RMSE
- fit the model

**Note:** The use of `parallelism = 128` in `CrossValidator()` speeds up the runtime from about 12 minutes to about 6 minutes! Please be patient! Additionally, the red line that appears under the code chunk is *not* representative of an actual error, but rather staging outputs from the log.

In [30]:
# build parameter grid
paramGrid = ParamGridBuilder() \
            .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .build()


# set up cross validation with pipeline
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5,
                          seed = 44,
                          parallelism = 128)

# fit the model using cross-validation to choose the best model
cvModel = crossval.fit(power_ML)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [17]:
my_list = []

for i in range(len(paramGrid)):
    my_list.append([cvModel.avgMetrics[i], paramGrid[i].values()])
    
my_list.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list

[[2175.025614452874, dict_values([0.25, 0.5])],
 [2175.0335187645323, dict_values([0.5, 0.25])],
 [2175.0335454632104, dict_values([0.5, 0.05])],
 [2175.0341016574052, dict_values([0.5, 0.1])],
 [2175.0349079585444, dict_values([0.05, 0.5])],
 [2175.035462853383, dict_values([0.25, 0.1])],
 [2175.035512165499, dict_values([0.1, 0.95])],
 [2175.035587292835, dict_values([0.05, 0.98])],
 [2175.0356041252753, dict_values([0.05, 0.9])],
 [2175.0356236594253, dict_values([0.05, 0.95])],
 [2175.035731007239, dict_values([0.05, 0.99])],
 [2175.035734665818, dict_values([0.05, 1.0])],
 [2175.035803359566, dict_values([0.05, 0.25])],
 [2175.035833255783, dict_values([0.1, 0.0])],
 [2175.035841619817, dict_values([0.1, 0.1])],
 [2175.0358417832153, dict_values([0.25, 0.0])],
 [2175.035842577646, dict_values([0.05, 0.75])],
 [2175.03585093693, dict_values([0.05, 0.0])],
 [2175.035880590475, dict_values([0.0, 0.98])],
 [2175.0358805904757, dict_values([0.0, 0.25])],
 [2175.035880590477, dict_value

Thus, the best multiple linear regression model based on cross-validation with RMSE is one with a regularization parameter of 0.25 and an elastic net parameter of 0.5. This tells us that we have a combination of L1 and L2 penalties; hence, we are fitting an elastic net model! We can now print the intercept and coefficients of this best model, as well as the CV error (with its corresponding tuning parameters).

In [18]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel.bestModel.stages[-1].intercept)
print('Coefficients (in Features Order):', cvModel.bestModel.stages[-1].coefficients)

print(' ')

print("Best regParam:", cvModel.bestModel.stages[-1]._java_obj.getRegParam())
print("Best elasticNetParam:", cvModel.bestModel.stages[-1]._java_obj.getElasticNetParam())
print('Resulting Best CV RMSE:', round(min(cvModel.avgMetrics), 4))

Intercept: 17831.197607816746
Coefficients (in Features Order): [281.33759408416756,-508.5488523409969,-933.3422279345817,4380.973673101201,582.7712870595075,0.0,1670.6436195006297,1521.0143011644407,1488.0485558207433,1932.6899471239333,1411.9841811335202,1784.2129277383003,3556.7197890829834,2402.836302038806,373.59506055395997,-54.722433929871215,504.5217858788957]
 
Best regParam: 0.25
Best elasticNetParam: 0.5
Resulting Best CV RMSE: 2175.0256


We now report the training set RMSE, i.e., the RMSE on the full dataset, by using the fitted model as a transformer and evaluating on the entire training (full) dataset.

In [20]:
training_rmse = RegressionEvaluator(metricName = 'rmse').evaluate(cvModel.transform(power_ML))

print('Entire Training (Full) Dataset RMSE:', round(training_rmse, 5))

Entire Training (Full) Dataset RMSE: 2174.44967


The last step is to take the outputted transformations from the model (i.e., the predictions) and create a `residual` column defined as `label` - `prediction`. We will print out a resulting dataframe with the following columns:
- `residual`
- `label`
- `prediction`

In [21]:
# store previous model transformer in an object
tf_final = cvModel.transform(power_ML)

# create residual column
resids_label_preds = tf_final.withColumn("residual", col("label") - col("prediction"))
resids_label_preds.select("residual", "label", "prediction").toPandas()

,residual,label,prediction
0,-695.215712,20240.96386,20936.179572
1,1431.166976,20131.08434,18699.917364
2,1432.893079,19668.43373,18235.540651
3,1284.964277,18899.27711,17614.312833
4,1426.591996,18442.40964,17015.817644
...,...,...,...
47169,2469.850984,14780.31212,12310.461136
47170,2647.362365,14428.81152,11781.449155
47171,2633.732742,13806.48259,11172.749848
47172,2794.162807,13512.60504,10718.442233


## **Streaming**
In this part of the project, we will be using the model created in the previous section with streaming data!

### **Reading a Stream**
In my `jupyterhub` main directory, I have created a folder called `csv_files_final`. This folder will be used to read in the stream in the form of `.csv` files. 

The code below sets up the schema for the stream by using the schema from the original data as we did in the Homework 10 assignment.

In [22]:
myschema = power_ML.schema
power_stream = spark.readStream.option("header", "true").schema(myschema).csv("csv_files_final")

### **Transformations and Aggregations**

First, with the stream, we use the model transformer to obtain predictions from the incoming data. This can be accomplished by creating an object very similar to `tf_final`. The only difference is that rather than supplying `power_ML` -- our data object -- to the `transform()` method, we will supply the stream object -- `power_stream`. The newly created object will be called `power_transform`, and we can use this to create a `residual` column.

In [23]:
# use model transformer with stream
power_transform = cvModel.transform(power_stream)

# create residual column
pwr_tf_resids = power_transform.withColumn("residual", col("label") - col("prediction")) \
                               .select("residual", "label", "prediction")

Next, we can use the stream again, with another transformation on the original stream, by modifying the response variable (`Power_Zone_3`) to be called `label`. This will be done using the `.withColumnRenamed()` method, rather than an SQL Transformer, because we want to keep all the original variables as well.

In [24]:
label_rename = power_stream.withColumnRenamed("Power_Zone_3", "label")

We can now join the above previous transform with this stream based on the `label` variable. This will be done with an inner join via the `.join()` method.

In [25]:
stream_join = pwr_tf_resids.join(label_rename, "label", "inner")

### **Writing Step**
We are now ready to write the `stream_join`. We will write it to the console using the `append` output mode. The code below will start the query.

In [33]:
query = stream_join.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19633.35423|-2277.2771074885422|21910.631337488543|      26.01|    72.0|     4.919|                588.5|        119.1| 34342.64151| 20923.33685|    8|  10|
|25331.17409| -979.1840641396084| 26310.35815413961|      19.91|    79.8|      4.92|                67.03|         70.6| 44934.29508| 26990.71207|    5|  19|
|16730.05025| 131.37553991079767|16598.674710089203|      16.43|   56.76|     0.082|                722.0|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16197.81818| 195.92278184944917|16001.895398150551|      14.31|    89.3|     0.068|                0.059|        0.156| 24608.65447| 12731.97556|    4|   4|
|24585.64824|  863.5604592333875| 23722.08778076661|      13.94|   63.93|     0.075|                0.055|        0.111| 40777.62712| 24995.74468|    2|  21|
|    15424.0|-45.211934184913844|15469.211934184914|      16.78|    70.4|     0.081|                0.022|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15886.45161|-1794.1553541388057|17680.606964138806|      13.25|    83.4|     0.078|                229.0|        217.1| 33653.10638| 20612.19512|    3|  11|
|25159.35484|  543.6870959447733|24615.667744055227|      13.52|   53.24|     0.084|                0.066|        0.115| 42967.14894| 25763.41463|    3|  20|
|23948.86432| 1484.3403262177635|22464.523993782237|      13.38|   58.95|     0.085|                0.051|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25339.35484|-677.9741550978797| 26017.32899509788|      11.78|    78.5|     0.083|                 7.02|        6.512| 44780.93617| 26484.14634|    3|  19|
|17135.27638| 315.8419007521261|16819.434479247873|      13.84|    72.4|     0.073|                146.1|        145.1| 31545.76271| 18751.36778|    2|  11|
| 42940.9205| 6780.887419429491| 36160.03308057051|      29.54|   47.65|     0.068|                0.256|        0.204

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|29811.15987| 2083.429798457386|27727.730071542614|      24.87|    88.6|     4.917|                 0.08|        0.133| 37558.26859| 26291.02429|    8|   0|
|16628.36364|-477.1017104154853|17105.465350415485|      13.57|    84.9|     4.923|                0.073|        0.159| 25358.88052| 13850.10183|    4|   4|
|33344.20063|  778.405686340695|32565.794943659304|      23.58|   64.55|     4.902|                 0.08|        0.089

...

-------------------------------------------
Batch: 12
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16679.87743| -273.3123491750048|16953.189779175005|       21.0|    78.2|     4.917|                0.091|        0.093| 36401.41593| 20881.49688|    9|  22|
|27138.80878|  1525.408540903085|25613.400239096914|      25.83|    88.6|     4.908|                0.059|        0.163| 34483.28524| 24223.02006|    8|   1|
| 13008.1459|-378.31820393880116|  13386.4641039388|      25.57|   42.88|     4.919|                460.7|      

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18692.05312|  3246.388513383068|15445.664606616932|       23.7|   62.15|     0.273|                 0.08|          0.1| 35018.76106| 21057.38046|    9|  23|
|10313.19838|-1132.4962560869444|11445.694636086944|      19.13|    79.3|      0.07|                29.97|        23.51|  20912.2623| 11238.39009|    5|   6|
|32374.15385|  5706.553660183959| 26667.60018981604|      22.72|   63.08|      4.92|                0.106|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20402.89157|  550.4650509495659|19852.426519050434|       19.2|    72.6|      0.07|                0.124|        0.093| 39304.61538| 32202.89256|   11|  18|
|17505.54217| 3571.2946367729855|13934.247533227015|      22.28|   50.37|     0.074|                360.2|        110.1| 32695.38462| 24024.79339|   11|  15|
|17936.05016|-3314.9121102636564|21250.962270263655|      24.12|    82.0|     4.921|                225.0|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9692.196879| -2883.959364429258|12576.156243429257|      14.04|   69.95|     0.086|                36.35|        35.69| 31793.15589| 26794.72231|   12|  16|
| 20776.9279|-358.05679232770854|21134.984692327707|      27.02|   53.92|     4.902|                0.113|        0.089| 28333.31853| 20942.34424|    8|   4|
|16840.62696|  -4194.62339225721| 21035.25035225721|      24.29|    89.3|     4.916|                0.069|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14692.34171|1348.6273707091732|13343.714339290827|       13.8|    72.1|     0.074|                0.088|          0.1|  22557.9661| 13272.94833|    2|   2|
|25347.46988|1066.8012648485128|24280.668615151488|      11.46|   53.89|     0.086|                0.059|        0.122| 41085.56962| 27114.89362|    1|  21|
|9311.884754| 1916.139355911204| 7395.745398088797|       7.26|    90.1|     0.082|                0.015|        0.12

## **Producing the Data**
At this point, the `power_streaming_data.csv` file should be uploaded into the main directory in `jupyterhub`. Additionally, the `Loring_FinalProject.py` file reads in the `power_streaming_data.csv` file and writes a loop that randomly samples some rows from this file, outputs them to a `.csv` (removing the indices), and pauses for 10 seconds in between outputting of datasets.

After running the above line of code to start the query, we can run the loop from the `Loring_FinalProject.py` file in a python console. Once we do this, we will see output here in this notebook right above the start of this section. Then the query will be stopped. Please see the video uploaded to Moodle within the Final Project assignment to see a recording of this process play out in real time.

In [34]:
query.stop()